# Tutorial 10: Using Claude for Principal Component Analysis
## Find lower-dimensional structure while preserving the assumptions and information loss

**Course:** IE 1171  
**Files used:** generated two-variable data and `winequality-red.csv`  
**Level 1:** Required core—standardization, covariance, eigenvectors, scores, loadings, and reconstruction  
**Level 2:** Optional deep dive—principal components regression and privacy limits

---

Claude will help coordinate the linear algebra and code. Your responsibility is to decide whether variables should be centered or scaled, verify the eigenvalue relationships, interpret loadings without inventing meaning, and quantify what is lost when dimensions are removed.

# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Explain PCA mathematically**
   - Connect centering, scaling, covariance, eigenvectors, and eigenvalues.
   - Derive the first principal component as a maximum-variance direction.
   - Distinguish loadings, scores, explained variance, and reconstruction error.
2. **Apply PCA carefully**
   - Verify a manual eigendecomposition against `sklearn.decomposition.PCA`.
   - Build a scree plot and choose a component count using evidence rather than a fixed slogan.
   - Interpret red-wine components using the supplied physicochemical variables.
3. **Judge the limits of compression**
   - Explain why scaling changes the PCA question.
   - Use a pipeline to avoid leakage in principal components regression.
   - Explain why lower dimension is not automatic anonymization or fairness.

# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define what PCA should preserve, why dimension reduction is useful, and what could be lost. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Plan centering/scaling, covariance, component selection, and reconstruction checks. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Compute PCA and apply it to the wine data while retaining checkable intermediate quantities. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Examine explained variance, loadings, reconstruction, interpretability, and privacy implications. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

## Statistical Reading

**Gareth James, Daniela Witten, Trevor Hastie, Robert Tibshirani, and Jonathan Taylor, _An Introduction to Statistical Learning with Applications in Python_ (ISLP)**

- **Chapter 6, Section 6.3.1, “Principal Components Regression”: printed pages 254–259 in the attached Python edition.**
- **Chapter 12, Section 12.2, “Principal Components Analysis”: printed pages 504–515 in the attached Python edition.**

Focus on:

- why correlated variables can be summarized by a smaller set of directions;
- the optimization definition of the first principal component;
- scores and loading vectors;
- proportion of variance explained;
- the difference between unsupervised PCA and supervised prediction;
- why component interpretation requires the original variable definitions.

## Ethical / Social-Good Reading

**Michael Kearns and Aaron Roth, _The Ethical Algorithm_**

- Chapter 3, **“Shopping with 300 Million Friends”** (pp. 116–118)
- Chapter 3, **“Shopping, Visualized”** (pp. 118–121)
- Chapter 3, **“A Different Kind of Cloud Computing”** (pp. 121–123)

Use these readings to connect representation and dimension reduction to the social effects of summarizing people, preferences, and relationships into compact numerical descriptions.


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Part | Purpose |
|---|---|
| **1. Geometry** | See PCA rotate a two-dimensional cloud. |
| **2. Linear algebra** | Calculate covariance, eigenvalues, eigenvectors, and scores. |
| **3. Verification** | Compare manual calculations with scikit-learn. |
| **4. Wine data** | Apply PCA to correlated physicochemical measurements. |
| **5. Interpretation** | Read scree plots, loadings, scores, and reconstruction. |
| **6. Level 2** | Use leakage-safe PCR and test privacy claims. |

## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook's normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect equations, assumptions, and concepts to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the details an AI collaborator can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run the response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: PCA as an Optimization, Rotation, and Approximation

Suppose $X\in\mathbb{R}^{n\times p}$ contains $n$ observations and $p$ numeric variables. PCA begins by centering each column:

$$
X_c=X-\mathbf{1}\bar x^T.
$$

If variables use incomparable units, standardize with $Z_{ij}=(X_{ij}-\bar X_j)/s_j$. PCA on $X_c$ uses the covariance matrix; PCA on $Z$ uses the correlation matrix. These answer different questions. Covariance PCA gives more influence to variables with larger variance or units, while correlation PCA gives each standardized variable variance one.

The sample covariance matrix is

$$
S=\frac{1}{n-1}X_c^TX_c.
$$

The first loading vector solves

$$
v_1=\arg\max_{\lVert v\rVert_2=1}\operatorname{Var}(X_cv)
=\arg\max_{\lVert v\rVert_2=1}v^TSv.
$$

The solution is the eigenvector of $S$ with largest eigenvalue: $Sv_k=\lambda_kv_k$. Later loading vectors maximize remaining variance while being orthogonal to earlier vectors. If $V=[v_1,\ldots,v_p]$, then $V^TV=I$ and the component score matrix is

$$
T=X_cV.
$$

Observation $i$ receives a score on each component; variable $j$ receives a loading on each component. Scores locate observations in component space. Loadings define the directions and show how original variables contribute. The sign of an eigenvector is arbitrary: $v_k$ and $-v_k$ describe the same axis, so software may flip all scores and loadings for a component without changing the PCA.

The eigenvalue $\lambda_k$ is the variance of component $k$. Proportion of variance explained is

$$
PVE_k=\frac{\lambda_k}{\sum_{j=1}^{p}\lambda_j},
\qquad
CPVE_q=\sum_{k=1}^{q}PVE_k.
$$

Keeping $q<p$ components produces the rank-$q$ reconstruction

$$
\hat X_c=T_qV_q^T=X_cV_qV_q^T.
$$

Among rank-$q$ linear approximations, PCA minimizes squared reconstruction error $\lVert X_c-\hat X_c\rVert_F^2=\sum_{k=q+1}^{p}(n-1)\lambda_k$. This optimality concerns variance and squared distance—not prediction, causality, fairness, or semantic usefulness.

### Questions you should be ready to answer

- Why must PCA center variables?
- When does scaling materially change the result?
- What is the difference between a score and a loading?
- Why can a component's sign flip with no change in meaning?
- What exactly is optimized by the first component and by a $q$-component reconstruction?

## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define what PCA should preserve, why dimension reduction is useful, and what could be lost.

# Level 1 — Required Core

# Part 1: Create a Two-Dimensional Example

The old tutorial used generated random data to make the geometry visible before applying PCA to the red-wine data. We keep that sequence: learn the rotation in two dimensions, verify the equations, and only then move to eleven variables.

## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Plan centering/scaling, covariance, component selection, and reconstruction checks.

## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Predict the Principal Direction

Imagine a scatterplot shaped like a long tilted ellipse.

1. Which direction should PC1 follow?
2. Where should PC2 point relative to PC1?
3. Which component should have larger variance?
4. What should happen to the sample mean after centering?
5. Would multiplying one variable by 1,000 change covariance PCA? Would it change correlation PCA?
6. Why is a component a direction rather than one original variable?

## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Compute PCA and apply it to the wine data while retaining checkable intermediate quantities.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Generate and Plot Correlated Data

```text
Write one Jupyter code cell that:

1. imports NumPy, pandas, and matplotlib;
2. creates rng = np.random.default_rng(1099);
3. creates 160 observations where x1 is standard normal and
   x2 = 0.85*x1 + Normal(0, 0.45);
4. stores them in toy_data with columns x1 and x2;
5. prints means, standard deviations, and the correlation matrix;
6. creates an equal-axis scatterplot with horizontal and vertical lines at the means;
7. fits no PCA model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(1099)
x1 = rng.normal(size=160)
x2 = 0.85 * x1 + rng.normal(0, 0.45, size=160)
toy_data = pd.DataFrame({"x1": x1, "x2": x2})

print("Means")
display(toy_data.mean().to_frame("mean"))
print("Standard deviations")
display(toy_data.std(ddof=1).to_frame("sd"))
print("Correlation")
display(toy_data.corr())

ax = toy_data.plot.scatter(x="x1", y="x2", alpha=0.7, figsize=(6, 6))
ax.axvline(toy_data["x1"].mean(), color="gray", linewidth=1)
ax.axhline(toy_data["x2"].mean(), color="gray", linewidth=1)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Correlated two-dimensional data")
plt.show()

# Part 2: Calculate PCA From the Covariance Matrix

For the toy data, center but do not standardize because the two variables were created on comparable scales. `numpy.linalg.eigh` is appropriate for the symmetric covariance matrix. It returns eigenvalues in ascending order, so they must be sorted from largest to smallest for PCA.

The covariance matrix is symmetric and positive semidefinite. Therefore its eigenvalues are nonnegative apart from tiny floating-point error, and its eigenvectors can be chosen orthonormal.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Perform PCA Manually

```text
Using toy_data:

1. center both columns and verify their means are approximately zero;
2. calculate the sample covariance as X_centered.T @ X_centered / (n - 1);
3. use np.linalg.eigh, then sort eigenvalues and eigenvectors descending;
4. name the sorted eigenvector matrix loadings_manual;
5. calculate scores_manual = X_centered @ loadings_manual;
6. print eigenvalues, loadings, explained-variance ratios, V.T @ V,
   and the covariance of the score columns;
7. verify numerically that S @ v equals lambda * v for each component.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2

In [ ]:
X_centered = toy_data - toy_data.mean()
n = len(X_centered)
S = X_centered.to_numpy().T @ X_centered.to_numpy() / (n - 1)

eigenvalues, eigenvectors = np.linalg.eigh(S)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
loadings_manual = eigenvectors[:, order]
scores_manual = X_centered.to_numpy() @ loadings_manual
explained_ratio_manual = eigenvalues / eigenvalues.sum()

print("Centered means:", X_centered.mean().round(12).to_dict())
display(pd.DataFrame(S, index=toy_data.columns, columns=toy_data.columns))
display(pd.DataFrame(loadings_manual, index=toy_data.columns,
                     columns=["PC1", "PC2"]))
print("Eigenvalues:", eigenvalues)
print("Explained ratios:", explained_ratio_manual)
print("V.T @ V")
display(pd.DataFrame(loadings_manual.T @ loadings_manual).round(12))
print("Score covariance")
display(pd.DataFrame(np.cov(scores_manual, rowvar=False)).round(12))

for j in range(2):
    np.testing.assert_allclose(S @ loadings_manual[:, j],
                               eigenvalues[j] * loadings_manual[:, j])
print("Verified both eigenvector equations.")

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Are the centered means numerically near zero?
- Are eigenvalues sorted largest first?
- Is $V^TV$ approximately the identity matrix?
- Are off-diagonal score covariances approximately zero?
- Does the first loading direction follow the long axis of the data?
- Does each eigenvalue equal the corresponding score variance?
- Could both loading signs be reversed without changing the answer?

# Part 3: Verify With `sklearn` and Reconstruct the Data

Scikit-learn stores loading vectors in `pca.components_` as rows rather than columns. Its transformed output contains the scores. A sign difference from the manual calculation is allowed; compare absolute loadings or align signs before comparing.

With only PC1, reconstruction projects every centered point onto a line. The discarded PC2 distances create reconstruction error.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Compare, Project, and Reconstruct

```text
Using toy_data and sklearn.decomposition.PCA:

1. fit a two-component PCA to the unstandardized toy data;
2. print components, explained variance, and explained-variance ratios;
3. compare absolute sklearn loadings with absolute manual loadings;
4. fit a one-component PCA and reconstruct the observations;
5. calculate mean squared reconstruction error across all matrix entries;
6. plot original points, one-component reconstructions, and faint segments
   joining each original point to its reconstruction;
7. use equal axis scaling.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3

In [ ]:
from sklearn.decomposition import PCA

pca_two = PCA(n_components=2).fit(toy_data)
scores_sklearn = pca_two.transform(toy_data)

print("Components (rows are loading vectors)")
display(pd.DataFrame(pca_two.components_, columns=toy_data.columns,
                     index=["PC1", "PC2"]))
print("Explained variance:", pca_two.explained_variance_)
print("Explained ratios:", pca_two.explained_variance_ratio_)
np.testing.assert_allclose(np.abs(pca_two.components_.T),
                           np.abs(loadings_manual), atol=1e-10)

pca_one = PCA(n_components=1).fit(toy_data)
one_scores = pca_one.transform(toy_data)
toy_reconstructed = pca_one.inverse_transform(one_scores)
reconstruction_mse = np.mean((toy_data.to_numpy() - toy_reconstructed) ** 2)
print("One-component reconstruction MSE:", reconstruction_mse)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(toy_data["x1"], toy_data["x2"], alpha=0.6, label="original")
ax.scatter(toy_reconstructed[:, 0], toy_reconstructed[:, 1],
           alpha=0.6, label="1-PC reconstruction")
for original, rebuilt in zip(toy_data.to_numpy(), toy_reconstructed):
    ax.plot([original[0], rebuilt[0]], [original[1], rebuilt[1]],
            color="gray", alpha=0.18)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend()
plt.show()

# Part 4: Load and Audit the Red-Wine Data

This tutorial uses the UCI red-wine quality data, which contains eleven physicochemical measurements and a `quality` score. PCA will use only the eleven predictors. `quality` may color a visualization or become a supervised response later, but it must not enter the unsupervised PCA feature matrix.

Expected predictors are fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, and alcohol.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Load and Verify `winequality-red.csv`

```text
Write one Jupyter code cell that:

1. loads winequality-red.csv, first trying sep=";" and then ordinary CSV
   if only one column is produced;
2. standardizes column names to lowercase words joined by underscores;
3. verifies the expected 11 predictor columns and quality;
4. prints shape, data types, missing counts, duplicate-row count,
   summary statistics, and quality counts;
5. creates wine_features with only the 11 predictors;
6. fits no model and drops no row.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4

In [ ]:
from pathlib import Path

wine_path = Path("winequality-red.csv")
if not wine_path.exists():
    raise FileNotFoundError("Place winequality-red.csv beside this notebook.")

wine = pd.read_csv(wine_path, sep=";")
if wine.shape[1] == 1:
    wine = pd.read_csv(wine_path)
wine.columns = (wine.columns.str.strip().str.lower()
                .str.replace(r"[^a-z0-9]+", "_", regex=True)
                .str.strip("_"))

wine_predictors = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "ph", "sulphates", "alcohol",
]
required = wine_predictors + ["quality"]
missing_columns = sorted(set(required) - set(wine.columns))
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Shape:", wine.shape)
display(wine[required].dtypes.to_frame("dtype"))
display(wine[required].isna().sum().to_frame("missing"))
print("Duplicate rows:", wine.duplicated().sum())
display(wine[required].describe().T)
display(wine["quality"].value_counts().sort_index().to_frame("count"))
wine_features = wine[wine_predictors].copy()

# Part 5: Standardize, Fit PCA, and Choose a Dimension

The wine variables have different units and variances, so Level 1 uses standardized features. For $p=11$ standardized variables, the total sample variance is approximately 11, and the eigenvalues sum to approximately 11.

A scree plot shows $\lambda_k$ or $PVE_k$ by component. Possible selection rules include an elbow, a cumulative-variance target, downstream cross-validation, reconstruction error, and interpretability. No threshold—80%, 90%, or otherwise—is universally correct. The retained dimension must fit the purpose and cost of lost information.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Fit and Diagnose Wine PCA

```text
Using complete rows of wine_features:

1. fit StandardScaler, then PCA with all components;
2. name the standardized matrix wine_z and fitted PCA wine_pca;
3. create tables of explained variance, PVE, and cumulative PVE;
4. verify the explained variances sum to approximately 11;
5. create a scree plot and cumulative-PVE plot;
6. report the smallest q reaching at least 90% cumulative PVE;
7. create a loading table with original variables as rows;
8. list the three largest absolute loadings for PC1 through PC3;
9. do not assign causal or chemical meaning automatically.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5

In [ ]:
from sklearn.preprocessing import StandardScaler

wine_complete = wine_features.dropna().copy()
wine_scaler = StandardScaler()
wine_z = wine_scaler.fit_transform(wine_complete)
wine_pca = PCA().fit(wine_z)
wine_scores = wine_pca.transform(wine_z)

pve = wine_pca.explained_variance_ratio_
cpve = np.cumsum(pve)
pca_summary = pd.DataFrame({
    "component": np.arange(1, len(pve) + 1),
    "eigenvalue": wine_pca.explained_variance_,
    "pve": pve,
    "cumulative_pve": cpve,
})
display(pca_summary)
print("Sum of eigenvalues:", wine_pca.explained_variance_.sum())
q_90 = int(np.argmax(cpve >= 0.90) + 1)
print("Smallest q reaching 90% cumulative PVE:", q_90)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pca_summary["component"], pca_summary["pve"], marker="o")
axes[0].set(xlabel="Component", ylabel="Proportion of variance explained",
            title="Wine PCA scree plot")
axes[1].plot(pca_summary["component"], pca_summary["cumulative_pve"], marker="o")
axes[1].axhline(0.90, color="gray", linestyle="--")
axes[1].set(xlabel="Number of components", ylabel="Cumulative PVE",
            title="Cumulative explained variance")
plt.tight_layout()
plt.show()

loading_table = pd.DataFrame(
    wine_pca.components_.T,
    index=wine_predictors,
    columns=[f"PC{i}" for i in range(1, len(wine_predictors) + 1)],
)
display(loading_table)
for component in ["PC1", "PC2", "PC3"]:
    print(component)
    display(loading_table[component].reindex(
        loading_table[component].abs().sort_values(ascending=False).index
    ).head(3).to_frame("loading"))

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Was `quality` excluded from the feature matrix?
- Were all scaling parameters fitted from the analyzed feature rows?
- Do the eigenvalues sum to approximately the number of standardized variables?
- Is cumulative PVE monotone and does it end at one?
- Which variables dominate PC1, PC2, and PC3 by absolute loading?
- Are variables with opposite loading signs contrasted along the component?
- Why can high PVE coexist with weak prediction of `quality`?

# Part 6: Scores, Loadings, and Reconstruction

A score plot can reveal clusters, gradients, or outliers, but these patterns are descriptive. Coloring by `quality` after PCA does not make PCA supervised: the component directions were fitted without `quality`. It also does not establish that quality causes the separation.

Reconstruction makes information loss measurable. In standardized space, the error for observation $i$ is $\lVert z_i-\hat z_i\rVert_2^2$. Variables and observations with large errors are poorly represented by the retained components.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Visualize Scores and Quantify Information Loss

```text
Using wine_pca, wine_z, wine_scores, q_90, and the matching quality rows:

1. plot PC1 versus PC2, colored by quality;
2. label axes with each component's PVE;
3. reconstruct wine_z using the first q_90 components;
4. calculate overall reconstruction MSE;
5. calculate reconstruction MSE by original variable and by observation;
6. display the five variables and five observations with largest error;
7. do not label score clusters as causal groups.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

# AI for Social Good: Compression and Deployment Both Decide What Gets Lost

PCA can make high-dimensional public-health, environmental, infrastructure, or nonprofit data easier to visualize and model. A technically elegant representation can still fail during deployment when it misses community needs, operational constraints, or rare but important cases.

Maximizing total variance does **not** guarantee preservation of:

- small or underserved groups;
- low-variance safety or health signals;
- variables that partners consider important even if they explain little statistical variance;
- meaningful distinctions that are averaged away in a compressed representation.

Before using principal components in a real system, ask:

1. Which observations and groups have the largest reconstruction errors?
2. Does the representation preserve the information needed for the actual intervention?
3. What do community or domain partners say is important that PCA may treat as low variance?
4. Could the components still reveal sensitive attributes through correlation?
5. Does the representation still work after the model leaves the original dataset and operating environment?

> **Social-good principle:** Statistical variance is not the same thing as social importance. Compression should be checked against the purpose, stakeholders, and deployment setting.


# Tutorial 10 Conclusion

You used Claude to coordinate a manual PCA calculation, verify covariance eigenvectors, compare with scikit-learn, standardize the red-wine predictors, interpret explained variance and loadings, and quantify reconstruction error.

The central lesson is that PCA rotates data toward maximum-variance directions and can approximate the original matrix with fewer dimensions. The result depends on units, scaling, sample, outliers, and the purpose for which information is being retained.

## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Examine explained variance, loadings, reconstruction, interpretability, and privacy implications.

# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

1. What changes when PCA is performed on covariance versus correlation?
2. Why are loading vectors constrained to unit length and orthogonal?
3. How does $Sv=\lambda v$ connect to component variance?
4. What is the difference between loading, score, eigenvalue, and PVE?
5. Why is loading sign arbitrary?
6. What does a scree plot show?
7. Why is 90% cumulative variance not a universal rule?
8. What does reconstruction error quantify?
9. Why can PCA be poor for predicting a response even when PVE is high?
10. Why does lower dimension not guarantee privacy?

# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dive

# Part 7: Principal Components Regression

Principal components regression (PCR) first replaces correlated predictors with the first $q$ component scores, then regresses a response on those scores. If $T_q=XV_q$, the regression model is

$$
y=T_q\gamma+\varepsilon.
$$

PCA chooses directions with high predictor variance, not directions most related to $y$. A low-variance direction can be highly predictive, so $q$ must be chosen using supervised cross-validation when the goal is prediction.

All preprocessing belongs inside the pipeline. If scaling and PCA use the full dataset before cross-validation, validation-fold information changes the learned component directions and produces leakage.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 7: Compare Leakage-Safe PCR With Linear Regression

```text
Using complete rows of the 11 wine predictors and quality:

1. create one 75/25 train-test split with random_state=1099;
2. build a Pipeline: StandardScaler, PCA, LinearRegression;
3. use 5-fold cross-validation on training rows only to compare
   n_components from 1 through 11 by negative RMSE;
4. fit the selected PCR pipeline to all training rows;
5. evaluate it once on the untouched test rows using RMSE and R-squared;
6. fit a baseline Pipeline with StandardScaler and LinearRegression and
   evaluate it on the same test rows;
7. return a comparison table;
8. do not call the higher test score universally better.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

# Part 8: PCA and Privacy—A Stress Test

PCA can make variables harder to name directly, but obscurity is not anonymization. The transformation is linear and may be approximately inverted when loadings, scaling parameters, and enough components are available. Even a few scores may preserve sensitive attributes through correlation.

### Privacy questions

1. If $T_q$, $V_q$, the means, and scales are released, what reconstruction is possible?
2. Which wine variables have the smallest reconstruction error at $q=2,4,6$?
3. Could a sensitive label be predicted from the retained scores?
4. Does deleting column names remove information encoded in the coordinates?
5. What additional privacy model—such as access control, aggregation, or differential privacy—would be needed for the actual release?

> PCA is a dimensionality-reduction method. It is not, by itself, a privacy guarantee.

# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 Look Back

1. Which $q$ minimized training-only cross-validated RMSE?
2. Did the first components preserve the directions most useful for predicting quality?
3. Why must scaling and PCA occur inside each cross-validation fold?
4. How did PCR compare with ordinary linear regression on the same test rows?
5. What can be reconstructed from component scores and loadings?
6. Why is low interpretability different from privacy?

# Sources and Course Resources

- James, Gareth, Daniela Witten, Trevor Hastie, and Robert Tibshirani. *An Introduction to Statistical Learning*, Section 6.3.1 and Section 12.2.
- UCI Machine Learning Repository. [Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality).
- Jolliffe, Ian T., and Jorge Cadima. “Principal Component Analysis: A Review and Recent Developments.” *Philosophical Transactions of the Royal Society A*, 2016.
